## Summary

In [1]:
import sqlite3
import os
import pandas as pd
import networkx as nx 
import numpy as np

script_dir = os.path.dirname(os.path.abspath('.'))
db_path = os.path.join(script_dir, "amici", "database", "supreme_court_docs.db")
db_path = os.path.abspath(db_path)  # Resolve any relative path components

In [2]:
if not os.path.exists(db_path):
    print(f"Database file not found at: {db_path}")
    print(f"Current working directory: {os.getcwd()}")
else:
    # Connect to SQLite database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    print(f"Successfully connected to SQLite database at {db_path}")

# Use this after connecting with either method above
try:
    # Sample query - adjust table name as needed
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    
    # Fetch and display results
    tables = cursor.fetchall()
    print("Tables in the database:")
    for table in tables:
        print(f"- {table[0]}")
        # Print all fields in table
        cursor.execute(f"PRAGMA table_info({table[0]})")
        fields = cursor.fetchall()
        print(f"  Fields in {table[0]}:")
        for field in fields:
            print(f"    - {field[1]} ({field[2]})")
except Exception as e:
    print(f"Error executing query: {e}")

Successfully connected to SQLite database at /Users/ejhall2/University of Michigan Dropbox/Galen Hall/Work/Amicus Briefs/amici/amici/database/supreme_court_docs.db
Tables in the database:
- documents
  Fields in documents:
    - document_id (INTEGER)
    - url (TEXT)
    - docket_url (TEXT)
    - date (TEXT)
    - date_formatted (DATE)
    - label (TEXT)
    - doc_title (TEXT)
    - blob (TEXT)
    - transcribed (BOOLEAN)
    - neededOCR (BOOLEAN)
    - complete_amici_list (BOOLEAN)
    - counsel_of_record (TEXT)
- sqlite_sequence
  Fields in sqlite_sequence:
    - name ()
    - seq ()
- dockets
  Fields in dockets:
    - docket_id (INTEGER)
    - document_id (INTEGER)
    - year (INTEGER)
    - number (INTEGER)
    - position (TEXT)
- amici
  Fields in amici:
    - amicus_id (INTEGER)
    - document_id (INTEGER)
    - name (TEXT)
    - category (TEXT)
    - merged_name (TEXT)
    - industry (TEXT)
- lawyers
  Fields in lawyers:
    - lawyer_id (INTEGER)
    - document_id (INTEGER)
   

In [3]:
# Get database stats
try:
    # Number of amici
    cursor.execute("SELECT COUNT(*) FROM amici")
    num_amici = cursor.fetchone()[0]
    print(f"Number of amici: {num_amici}")

    # Number of organizations/coalitions
    cursor.execute("SELECT COUNT(*) FROM amici WHERE category != 'individual' AND category != 'academic'")
    num_organizations = cursor.fetchone()[0]
    print(f"Number of organizations/coalitions: {num_organizations}")

    # Number of dockets
    cursor.execute("SELECT COUNT(*) FROM dockets")
    num_dockets = cursor.fetchone()[0]
    print(f"Number of dockets: {num_dockets}")
except Exception as e:
    print(f"Error fetching database stats: {e}")

# Save a spreadsheet with a single column containing all unique amicus merged_names
try:
    # Query to get all unique amicus merged_names where the 
    cursor.execute("SELECT DISTINCT merged_name FROM amici WHERE merged_name IS NOT NULL AND category = 'organization' ORDER BY merged_name")
    unique_merged_names = cursor.fetchall()
    
    # Create a DataFrame with the merged_names
    df_merged_names = pd.DataFrame([name[0] for name in unique_merged_names], columns=['merged_name'])
    
    # Save to Excel file
    output_path = "../amici/data/unique_amicus_merged_names.csv"
    df_merged_names.to_csv(output_path, index=False)
    
    print(f"Saved {len(df_merged_names)} unique amicus merged names to {output_path}")
    
except Exception as e:
    print(f"Error creating merged names spreadsheet: {e}")

Number of amici: 45159
Number of organizations/coalitions: 32057
Number of dockets: 13125
Saved 7620 unique amicus merged names to ../amici/data/unique_amicus_merged_names.csv


In [4]:
industries_path='../amici/analysis/classification/openai_classifications.csv'

df = pd.read_csv(industries_path)
mapping = df.set_index('interest_group').industry.to_dict()

# Update the amicus table to add industry field based on mapping
# cursor.execute("ALTER TABLE amici ADD COLUMN industry TEXT")

# Update industry for organizations using the mapping
for interest_group, industry in mapping.items():
    cursor.execute(
        "UPDATE amici SET industry = ? WHERE category = 'organization' AND merged_name = ?",
        (industry, interest_group)
    )

conn.commit()
conn.close()

## Make excel sheets

In [33]:
try:
    # Query to find documents with complete_amici_list=False
    cursor.execute("SELECT * FROM documents WHERE complete_amici_list=0")
    
    # Fetch and display results
    documents_with_appendix = cursor.fetchall()
    
    # Get column names from cursor description
    column_names = [description[0] for description in cursor.description]
    
    # Print count of documents
    print(f"Found {len(documents_with_appendix)} documents with amici list incomplete.")
    
    # Display column names and first few results
    if documents_with_appendix:
        print("\nColumn names:", column_names)
        print("\nFirst 5 documents with incomplete amici:")
        for doc in documents_with_appendix[:5]:
            print(doc)
except Exception as e:
    print(f"Error executing query: {e}")

# Create a file storing the blobs of all documents with complete_amici_list=False
try:
    cursor.execute("SELECT * FROM documents WHERE complete_amici_list=0")
    documents_with_appendix = cursor.fetchall()
    
    # Open a file to write the blobs
    with open("../amici/data/incomplete_amici_blobs.txt", "w") as f:
        for doc in documents_with_appendix:
            # Assuming the blob is in the first column (index 0)
            f.write(doc[7]+'\n')
    
    print("Blobs of incomplete amici documents have been written to incomplete_amici_blobs.txt")
except Exception as e:
    print(f"Error writing blobs to file: {e}")
# Close the database connection
finally:
    if conn:
        conn.close()
        print("Database connection closed.")

Found 1179 documents with amici list incomplete.

Column names: ['document_id', 'url', 'docket_url', 'date', 'date_formatted', 'label', 'doc_title', 'blob', 'transcribed', 'neededOCR', 'complete_amici_list', 'counsel_of_record']

First 5 documents with incomplete amici:
(4, 'http://www.supremecourt.gov/DocketPDF/22/22-535/252021/20230111151208528_22-506and22-535tsacLawyersCommitteeForCivilRightsUnderLaw.pdf', 'www.supremecourt.gov/search.aspx?filename=/docket/docketfiles/html/public/22-535.html', 'Jan 11 2023', '2023-01-11', 'Brief amici curiae of Lawyers’ Committee For Civil Rights Under Law and 21 Other Organizations filed (also in 22-506).  VIDED.  (Distributed)', 'Main Document', 'SUPREMECOURT/www.supremecourt.gov/DocketPDF/22/22-535/252021/20230111151208528_22-506and22-535tsacLawyersCommitteeForCivilRightsUnderLaw.pdf', 1, 0, 0, 'Damon Hewitt')
(29, 'http://www.supremecourt.gov/DocketPDF/19/19-1392/185243/20210729123007530_41063%20pdf%20Pierce.pdf', 'www.supremecourt.gov/search.as